In [ ]:
"""
Created on Wed Jul 02 15:07 2025

Look at the distribution of the weights to make a figure out of it

@author: Clara Burgard

"""

In [16]:
import xarray as xr
import numpy as np
from tqdm.notebook import tqdm
import seaborn as sns
import multimelt.useful_functions as uf
import os
import scipy.io
import matplotlib.pyplot as plt


In [4]:
sns.set_context('paper')

In [5]:
%matplotlib qt5

QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-burgardc'


READ IN DATA

In [12]:
home_path = '/bettik/burgardc/'
plot_path = '/bettik/burgardc/PLOTS/summer_paper_plots/'
outputpath_GL = '/bettik/burgardc/DATA/SUMMER_PAPER/processed/GL_FLUX/'
outputpath_weights = '/bettik/burgardc/DATA/SUMMER_PAPER/processed/ANALYSIS/'
inputpath_raw = '/bettik/burgardc/DATA/SUMMER_PAPER/raw/'
inputpath_data=home_path+'/DATA/SUMMER_PAPER/interim/'
inputpath_weights = '/bettik/burgardc/DATA/SUMMER_PAPER/processed/ANALYSIS/'


In [7]:
inputpath_mask = home_path+'/DATA/SUMMER_PAPER/interim/ANTARCTICA_IS_MASKS/BedMachine_4km/'
file_isf_orig = xr.open_dataset(inputpath_mask+'BedMachinev2_4km_isf_masks_and_info_and_distance_oneFRIS.nc')
nonnan_Nisf = file_isf_orig['Nisf'].where(np.isfinite(file_isf_orig['front_bot_depth_max']), drop=True).astype(int)
file_isf_nonnan = file_isf_orig.sel(Nisf=nonnan_Nisf)
rignot_isf = file_isf_nonnan.Nisf.where(np.isfinite(file_isf_nonnan['isf_area_rignot']), drop=True)
file_isf = file_isf_nonnan.sel(Nisf=rignot_isf)

In [8]:
# make the domain a little smaller to make the computation even more efficient - file isf has already been made smaller at its creation
map_lim = [-3000000,3000000]

In [9]:
BedMachine_orig = xr.open_dataset(inputpath_data+'BedMachine_v2_aggregated4km_allvars.nc')
file_BedMachine = uf.cut_domain_stereo(BedMachine_orig, map_lim, map_lim)
file_isf_conc = file_BedMachine['isf_conc']

grid_cell_area_file = xr.open_dataset(inputpath_data+'gridarea_ISMIP6_AIS_4000m_grid.nc').sel(x=file_isf.x,y=file_isf.y)
true_grid_cell_area = grid_cell_area_file['cell_area'].drop('lon').drop('lat')
cell_area_weight = true_grid_cell_area/(4000 * 4000)

lon = file_isf.longitude
lat = file_isf.latitude

xx = file_isf.x
yy = file_isf.y
dx = (xx[2] - xx[1]).values
dy = (yy[2] - yy[1]).values
grid_cell_area_const = abs(dx*dy)  
grid_cell_area_weighted = file_isf_conc * grid_cell_area_const * cell_area_weight

isf_stack_mask = uf.create_stacked_mask(file_isf['ISF_mask'], file_isf.Nisf, ['y','x'], 'mask_coord')


In [10]:
sorted_isf_rignot = [11,69,43,28,12,57,
                     70,44,29,13,58,71,45,30,14,
                     59,72,46,
                     31,
                     15,61,73,47,32,16,48,33,17,62,49,34,18,63,74,
                     50,35,19,64,
                     10,
                     36,20,65,51,37,
                     22,38,52,23,66,53,39,24,
                     67,40,54,75,25,41,
                     26,42,55,68,60,27]

In [13]:
weight_2300_file = xr.open_dataset(inputpath_weights + 'bayesian_weights_davison_varying_combined_2300_withoutGISS.nc')

In [24]:
bay_weights = weight_2300_file['bay_weights']

In [28]:
sens_weights = xr.DataArray(data=np.array([0.11,
                                           0.24,
                                           0.03,
                                           0.10,
                                           0.10,
                                           0.10,
                                           0.10,
                                           0.24,
                                           0.47,
                                           0.41,
                                           0.12,
                                           0.43,
                                           0.39,
                                           0.05]), dims=['model']).assign_coords({'model': 
                                                                                  ['ACCESS-CM2','ACCESS-ESM1-5','CanESM5',
                                                                                   'CESM2','CESM2-WACCM','CNRM-CM6-1','CNRM-ESM2-1',
                                                                                   'GFDL-CM4','GFDL-ESM4','GISS-E2-1-H', 'IPSL-CM6A-LR',
                                                                                   'MPI-ESM1-2-HR','MRI-ESM2-0','UKESM1-0-LL']})

In [29]:
model_2300 = ['ACCESS-CM2','ACCESS-ESM1-5','CanESM5','CESM2-WACCM', 'IPSL-CM6A-LR','MRI-ESM2-0','UKESM1-0-LL'] #,'GISS-E2-1-H'

In [48]:
ddim = 'm' #'param','model','m'


f = plt.figure()
f.set_size_inches(8.25*1.2, 8.25*1.2)

ax={}

leg_hdl = []

i = 0

weight = bay_weights * sens_weights
weight['param'] = ['linear','quadratic Ant slope','quadratic local slope','plume','box','neural network']
weight['m'] = ['1','3','5']

for kisf in tqdm(sorted_isf_rignot):
    
    ax[i] = f.add_subplot(8,8,i+1)
    
    weight_norm_kisf = weight.sel(Nisf=kisf, model=model_2300) / weight.sel(Nisf=kisf).sum(['param','model','m'])

    if ddim == 'param':
        weight_ddim = weight_norm_kisf.stack(stack_dim=['model','m'])
    elif ddim == 'model':
        weight_ddim = weight_norm_kisf.stack(stack_dim=['param','m'])
    elif ddim == 'm':        
        weight_ddim = weight_norm_kisf.stack(stack_dim=['param','model'])

    bins = weight_ddim[ddim]
    _, bin_edges = np.histogram(weight_ddim.values, bins=np.arange(len(bins)), weights=weight_ddim.values)

    hist = weight_ddim.sum('stack_dim') # / weight_ddim.sum()

    ax[i].bar(weight_ddim[ddim], hist,  edgecolor='None')#width=1,

    #ax[i].set_title(ano_choice.sel(Nisf=kisf))

    #if kisf == 23:
    #    ax[i].set_title('Tracy Tremenchus')
    #elif kisf == 24:#
    #    ax[i].set_title('Conger/Glenzer')
    #elif kisf == 110:
    #    ax[i].set_title('Ekström')
    #else:
    ax[i].set_title(str(file_isf['isf_name'].sel(Nisf=kisf).values))
    #if file_isf['isf_name'].sel(Nisf=kisf).values not in ['Larsen B', 'Wordie']:
    #    ax[i].set_ylim(0,mass_balance_simu_anoISMIP.sel(Nisf=kisf).max().values)
    #ax[i].axvline(x=30, c='k', linestyle='--')
    
    if i < 56:
        ax[i].set_xticklabels('')
    if i not in [0,8,16,24,32,40,48,56]:
        ax[i].set_yticklabels('')
        
    i = i+1

    plt.xticks(rotation=90)
#f.legend()
#f.subplots_adjust(bottom=0.05, wspace=0.1)

f.tight_layout()
sns.despine()
f.savefig(plot_path + 'histo_weights_per_iceshelf_2300_'+ddim+'.pdf')

  0%|          | 0/64 [00:00<?, ?it/s]

In [52]:
weight.sel(model=model_2300).stack(stack_dim=['model','m','Nisf']).sum('stack_dim')

<xarray.DataArray (param: 6)>
array([1.29111878, 1.78219843, 1.1347619 , 1.64186801, 1.53119956,
       1.80450145])
Coordinates:
  * param       (param) <U21 'linear' 'quadratic Ant slope' ... 'neural network'
    metrics     object ...
    box_nb_tot  int64 ...
    config      int64 ...

In [65]:
f = plt.figure()
f.set_size_inches(8.25/2, 8.25/2)
    
#weight_norm_kisf = weight.sel(model=model_2300) / weight.sum(['param','model','m'])

hist = weight.sel(model=model_2300).sum(['Nisf','model','m'])
plt.bar(weight['param'], hist,  edgecolor='None')
plt.xticks(rotation=90)
f.tight_layout()
sns.despine()
f.savefig(plot_path + 'histo_weights_2300_param_summedallisf.pdf')

In [64]:
weight

<xarray.DataArray (Nisf: 64, model: 7, param: 6, m: 3)>
array([[[[2.49929144e-04, 1.58157616e-04, 1.71097049e-04],
         [1.20194079e-03, 1.12101426e-03, 1.13833609e-03],
         [1.11264653e-03, 9.75089670e-04, 1.00026982e-03],
         [8.32635746e-04, 6.55471787e-04, 6.84274210e-04],
         [1.19809826e-03, 1.11260570e-03, 1.13059576e-03],
         [1.21562209e-03, 1.15911972e-03, 1.17279096e-03]],

        [[3.79460332e-04, 2.29214482e-04, 2.49856611e-04],
         [2.60438847e-03, 2.40762233e-03, 2.44836037e-03],
         [2.35805639e-03, 2.03529649e-03, 2.09305056e-03],
         [1.52284732e-03, 1.14719595e-03, 1.20624054e-03],
         [2.61898960e-03, 2.43821146e-03, 2.47662213e-03],
         [2.59184944e-03, 2.38323957e-03, 2.42568321e-03]],

        [[3.93126361e-05, 2.32214560e-05, 2.54053013e-05],
         [3.21171411e-04, 2.92813504e-04, 2.98443382e-04],
         [2.84909459e-04, 2.42099549e-04, 2.49605185e-04],
         [1.57218495e-04, 1.13748497e-04, 1.20393907e-04],
         [3.27284619e-04, 3.04581049e-04, 3.09397943e-04],
         [3.28649190e-04, 3.07705396e-04, 3.12263480e-04]],
...
        [[1.16665648e-03, 7.09896288e-04, 7.26138870e-04],
         [7.82554361e-04, 1.19124223e-03, 1.18672278e-03],
         [1.13274937e-03, 6.40156729e-04, 6.56199776e-04],
         [5.09040350e-04, 1.04104141e-03, 1.02830535e-03],
         [8.73511643e-04, 1.20593821e-03, 1.20474991e-03],
         [1.04252578e-03, 1.17515916e-03, 1.18087982e-03]],

        [[3.85575412e-03, 2.48699684e-03, 2.53963065e-03],
         [2.34715934e-03, 3.80768454e-03, 3.78629012e-03],
         [3.74443514e-03, 2.20171160e-03, 2.25431153e-03],
         [1.48589910e-03, 3.23042352e-03, 3.18528546e-03],
         [2.63285603e-03, 3.89233270e-03, 3.88083518e-03],
         [3.25927133e-03, 3.87017502e-03, 3.88318754e-03]],

        [[4.91765029e-04, 3.10811804e-04, 3.17575627e-04],
         [3.02440410e-04, 4.88750523e-04, 4.86058243e-04],
         [4.74637208e-04, 2.71597037e-04, 2.78303631e-04],
         [1.81586930e-04, 4.05252091e-04, 3.99288227e-04],
         [3.53223795e-04, 5.01531926e-04, 5.00632566e-04],
         [4.18728381e-04, 4.95896315e-04, 4.97601667e-04]]]])
Coordinates:
  * Nisf        (Nisf) int64 10 11 12 13 14 15 16 17 ... 68 69 70 71 72 73 74 75
  * param       (param) <U21 'linear' 'quadratic Ant slope' ... 'neural network'
  * model       (model) object 'ACCESS-CM2' 'ACCESS-ESM1-5' ... 'UKESM1-0-LL'
  * m           (m) <U1 '1' '3' '5'
    metrics     object ...
    box_nb_tot  int64 ...
    config      int64 ...
    ID_IMBIE    (Nisf) int64 66 124 132 5 13 22 31 38 ... 127 1 8 18 26 55 109